# 03 · Evaluation

Aggregates the JSON files emitted by `slurm/ablation.sbatch` (or local equivalent runs of `scripts/evaluate.py`) into the comparison table published in [`evals/results.md`](../evals/results.md).

Run order:
1. Build all three indexes (CLIP-L/14, SigLIP-2 base, SigLIP-2 large).
2. Run `scripts/evaluate.py` against each (the SLURM ablation does this).
3. Run this notebook to compute the table and the recall-curve figure.

In [ ]:
%matplotlib inline
import json
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns

sns.set_theme(style="whitegrid", context="talk")
plt.rcParams["figure.dpi"] = 110

EVAL_DIR = Path("../evals")

In [ ]:
result_files = sorted(EVAL_DIR.glob("results_*.json"))
results = []
for path in result_files:
    with path.open() as f:
        data = json.load(f)
    row = {
        "config": path.stem.replace("results_", ""),
        "encoder": data["encoder"],
        "dim": data["encoder_dim"],
        "n_corpus": data["n_corpus"],
        "n_queries": data["metrics"]["n_queries"],
    }
    row.update(data["metrics"]["recall"])
    row["mrr"] = data["metrics"]["mean_reciprocal_rank"]
    row["latency_p50_ms"] = data["metrics"]["latency_ms"]["p50"]
    row["latency_p95_ms"] = data["metrics"]["latency_ms"]["p95"]
    results.append(row)

df = pd.DataFrame(results)
df.set_index("config")

## Recall@K curves

Each encoder gets a curve. The shape matters: a curve that climbs steeply between K=1 and K=10 says the system has a few good candidates near the top but is missing the gold target — which is the case for compositional queries that match many plausible faces.

In [ ]:
k_values = [1, 5, 10, 50]
fig, ax = plt.subplots(figsize=(9, 5.5))

for _, row in df.iterrows():
    recalls = [row[f"recall@{k}"] for k in k_values]
    ax.plot(k_values, recalls, marker="o", linewidth=2, label=row["encoder"])

ax.set_xscale("log")
ax.set_xticks(k_values)
ax.set_xticklabels(k_values)
ax.set_xlabel("K (top-K retrieved)")
ax.set_ylabel("Recall@K")
ax.set_title(f"Retrieval recall on {df['n_queries'].iloc[0]} synthesized queries")
ax.legend()
ax.set_ylim(0, 1)
plt.tight_layout()
plt.savefig(EVAL_DIR / "figures" / "recall_curves.png", bbox_inches="tight")
plt.show()

## Latency-vs-quality

The right encoder isn't always the highest-recall one. For latency-sensitive deployments (interactive UI, ms-scale SLO), SigLIP-2 base is usually the right call. For batch eval or non-interactive search, SigLIP-2 large pays for itself.

In [ ]:
fig, ax = plt.subplots(figsize=(8, 5))
for _, row in df.iterrows():
    ax.scatter(row["latency_p95_ms"], row["recall@1"], s=120)
    ax.annotate(row["encoder"], (row["latency_p95_ms"], row["recall@1"]),
                xytext=(8, 4), textcoords="offset points", fontsize=10)
ax.set_xlabel("p95 latency (ms)")
ax.set_ylabel("Recall@1")
ax.set_title("Quality vs latency (interactive search regime)")
plt.tight_layout()
plt.savefig(EVAL_DIR / "figures" / "latency_vs_recall.png", bbox_inches="tight")
plt.show()

## Markdown table for the report

Copy-pasteable into [`evals/results.md`](../evals/results.md). Regenerate any time the eval is rerun.

In [ ]:
report_cols = ["recall@1", "recall@5", "recall@10", "recall@50", "mrr", "latency_p50_ms", "latency_p95_ms"]
report = df.set_index("encoder")[report_cols]
print(report.to_markdown(floatfmt=".2f"))